# Orders Feature Engineering

## Objective

This notebook creates business-oriented features from the cleaned `orders` dataset.

These engineered features will be used for:

- Exploratory Data Analysis (EDA)
- SQL Analytics
- Power BI Dashboard
- Business Insights & Recommendations

The engineered dataset produced in this notebook will be exported for downstream analysis.

In [5]:
import pandas as pd
import numpy as np

In [4]:
orders = pd.read_csv(
    "../../03_Python_ETL/Output/orders_clean.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ]
)

### Feature 1 — Order Approval Time
**Business Objective**

Measure how long it takes for an order to be approved after the customer places it.

This is an operational KPI used by e-commerce companies.

In [15]:
#Formula using Python Implementation
orders["approval_time_hours"] = (
    (
        orders["order_approved_at"]
        - orders["order_purchase_timestamp"]
    ).dt.total_seconds() / 3600
)

In [16]:
# Validation : Checking first few rows
orders[
    [
        "order_purchase_timestamp",
        "order_approved_at",
        "approval_time_hours",
    ]
].head()

,order_purchase_timestamp,order_approved_at,approval_time_hours
0,2017-10-02 10:56:33,2017-10-02 11:07:15,0.178333
1,2018-07-24 20:41:37,2018-07-26 03:24:27,30.713889
2,2018-08-08 08:38:49,2018-08-08 08:55:23,0.276111
3,2017-11-18 19:28:06,2017-11-18 19:45:59,0.298056
4,2018-02-13 21:18:39,2018-02-13 22:20:29,1.030556


In [17]:
#Check summary statistics
orders["approval_time_hours"].describe()

count    99281.000000
mean        10.419094
std         26.038004
min          0.000000
25%          0.215000
50%          0.343333
75%         14.580833
max       4509.180556
Name: approval_time_hours, dtype: float64

In [18]:
#Checking for negative values
orders.loc[
    orders["approval_time_hours"] < 0,
    [
        "order_purchase_timestamp",
        "order_approved_at",
        "approval_time_hours",
    ],
]

,order_purchase_timestamp,order_approved_at,approval_time_hours


In [32]:
# Checking missing values: These are the orders which are not approved
orders["approval_time_hours"].isna().sum()

160

### approval_time_hours

**Source Table**
orders

**Feature Type**
Numerical (Continuous)

**Business Definition**
Time taken between order placement and order approval.

**Formula**
(order_approved_at - order_purchase_timestamp) in hours

**Business Value**
Measures payment and order approval efficiency. Useful for identifying operational bottlenecks and analyzing their relationship with cancellations, delivery performance, and customer satisfaction.

**Null Logic**
Null when an order was never approved.

**Unit**
Hours

### Feature 2 — Processing Time
 **Business Objective**

Measure how long it takes the seller to prepare and hand over an approved order to the logistics carrier.

This is one of the most important operational KPIs in e-commerce because it evaluates seller fulfillment efficiency.

In [33]:
#Formula using Python Implementation
orders["processing_time_hours"] = (
    (
        orders["order_delivered_carrier_date"]
        - orders["order_approved_at"]
    ).dt.total_seconds() / 3600
)

In [34]:
#Validation by Checking the first few rows
orders[
    [
        "order_approved_at",
        "order_delivered_carrier_date",
        "processing_time_hours",
    ]
].head()

,order_approved_at,order_delivered_carrier_date,processing_time_hours
0,2017-10-02 11:07:15,2017-10-04 19:55:00,56.795833
1,2018-07-26 03:24:27,2018-07-26 14:31:00,11.109167
2,2018-08-08 08:55:23,2018-08-08 13:50:00,4.910278
3,2017-11-18 19:45:59,2017-11-22 13:39:59,89.900000
4,2018-02-13 22:20:29,2018-02-14 19:46:34,21.434722


In [35]:
#Summary statistics
orders["processing_time_hours"].describe()

count    97644.000000
mean        67.320916
std         85.186249
min      -4109.256111
25%         21.012222
50%         43.641528
75%         85.931250
max       3018.301667
Name: processing_time_hours, dtype: float64

In [37]:
# Missing values (Orders that were cancelled, unavailable, or never shipped will have NaN because there is no carrier handoff date)
orders["processing_time_hours"].isna().sum()

1797

In [45]:
orders.loc[
    orders["processing_time_hours"] < 0,
    [
        "order_approved_at",
        "order_delivered_carrier_date",
        "processing_time_hours",
    ],
]

,order_approved_at,order_delivered_carrier_date,processing_time_hours
15,2018-06-12 23:31:02,2018-06-11 14:54:00,-32.617222
64,2018-04-24 18:25:22,2018-04-23 19:19:14,-23.102222
199,2018-07-26 23:31:53,2018-07-24 12:57:00,-58.581389
210,2018-07-23 12:31:53,2018-07-23 12:24:00,-0.131389
415,2018-07-27 23:31:09,2018-07-24 14:03:00,-81.469167
...,...,...,...
99091,2018-07-05 16:17:59,2018-07-05 14:11:00,-2.116389
99230,2018-07-05 16:32:52,2018-07-03 12:57:00,-51.597778
99266,2018-02-04 23:31:46,2018-01-31 18:11:58,-101.330000
99377,2018-04-24 19:26:10,2018-04-23 17:18:40,-26.125000


### Feature 3: transit_time_days
 **Business Objective**

Measure the time taken by the logistics network to transport an order from the seller to the customer.

In [50]:
#Formula using Python Implementation
orders["transit_time_days"] = (
    (
        orders["order_delivered_customer_date"]
        - orders["order_delivered_carrier_date"]
    ).dt.total_seconds() / (24 * 3600)
)

In [51]:
orders["transit_time_days"].head()

0     6.062650
1    12.039410
2     9.178113
3     9.450498
4     1.937824
Name: transit_time_days, dtype: float64

In [52]:
orders["transit_time_days"].describe()

count    96475.000000
mean         9.330547
std          8.760122
min        -16.096169
25%          4.099948
50%          7.099769
75%         12.029115
max        205.190972
Name: transit_time_days, dtype: float64

In [53]:
orders["transit_time_days"].isnull().sum()

2966

In [54]:
(orders["transit_time_days"] < 0).sum()

23

### Feature 4: delivery_time_days
**Business Objective**

Measure the total time taken to deliver an order from the moment it was placed until it reached the customer.

This is one of the most important KPIs in e-commerce because it represents the customer's end-to-end waiting time.

In [55]:
#Formula using Python Implementation
orders["delivery_time_days"] = (
    (
        orders["order_delivered_customer_date"]
        - orders["order_purchase_timestamp"]
    ).dt.total_seconds() / (24 * 3600)
)

In [57]:
orders[["delivery_time_days"]].head()

,delivery_time_days
0,8.436574
1,13.782037
2,9.394213
3,13.208750
4,2.873877


In [59]:
orders[["delivery_time_days"]].describe()

,delivery_time_days
count,96476.000000
mean,12.558702
std,9.546530
min,0.533414
25%,6.766403
50%,10.217755
75%,15.720327
max,209.628611


In [60]:
orders["delivery_time_days"].isnull().sum()

2965

In [61]:
(orders["delivery_time_days"] < 0).sum()

0

### Feature 5: estimated_delivery_days
**Business Objective**

Measure the number of days promised to the customer for delivery at the time the order was placed.

Unlike the previous features, this is not an actual duration—it represents Olist's delivery commitment.

In [63]:
#Formula using Python Implementation
orders["estimated_delivery_days"] = (
    (
        orders["order_estimated_delivery_date"]
        - orders["order_purchase_timestamp"]
    ).dt.total_seconds() / (24 * 3600)
)

In [64]:
orders["estimated_delivery_days"].head()

0    15.544063
1    19.137766
2    26.639711
3    26.188819
4    12.112049
Name: estimated_delivery_days, dtype: float64

In [65]:
orders["estimated_delivery_days"].describe()

count    99441.000000
mean        23.767650
std          8.832371
min          1.648993
25%         18.331690
50%         23.240370
75%         28.424861
max        155.135463
Name: estimated_delivery_days, dtype: float64

In [66]:
orders["estimated_delivery_days"].isnull().sum()

0

In [67]:
(orders["estimated_delivery_days"] < 0).sum()

0

### Feature 6: delivery_delay_days

This is one of the most valuable KPIs in logistics and e-commerce analytics.

**Business Objective**

Measure how many days earlier or later an order was delivered compared to the promised delivery date.

This feature directly evaluates delivery performance against customer expectations.

In [68]:
#Formula using Python Implementation
orders["delivery_delay_days"] = (
    (
        orders["order_delivered_customer_date"]
        - orders["order_estimated_delivery_date"]
    ).dt.total_seconds() / (24 * 3600)
)

In [69]:
orders["delivery_delay_days"].describe()

count    96476.000000
mean       -11.179120
std         10.186113
min       -146.016123
25%        -16.244384
50%        -11.948941
75%         -6.390000
max        188.975081
Name: delivery_delay_days, dtype: float64

In [70]:
orders["delivery_delay_days"].isnull().sum()

2965

In [71]:
(orders["delivery_delay_days"] < 0).sum()

88649

In [72]:
(orders["delivery_delay_days"] > 0).sum()

7827

In [73]:
(orders["delivery_delay_days"] == 0).sum()

0

### Feature 7: delivered_on_time

This is one of the most useful features for SQL queries and Power BI dashboards.

**Business Objective**

Identify whether an order was delivered on or before the promised delivery date.

This converts the numerical delivery_delay_days into a simple business-friendly flag.

In [74]:
#Formula using Python Implementation
orders["delivered_on_time"] = (orders["delivery_delay_days"] <= 0)

In [76]:
orders["delivered_on_time"].value_counts()

delivered_on_time
True     88649
False    10792
Name: count, dtype: int64

In [77]:
orders["delivered_on_time"].value_counts(normalize=True) * 100

delivered_on_time
True     89.147334
False    10.852666
Name: proportion, dtype: float64

In [78]:
orders["delivered_on_time"].isnull().sum()

0

### Feature 8: purchase_month
**Business Objective**

Extract the month in which the customer placed the order.

In [82]:
#Python Implementation
orders["purchase_month"] = (
    orders["order_purchase_timestamp"]
    .dt.month_name()
)

orders["purchase_month_number"] = orders["order_purchase_timestamp"].dt.month

In [83]:
orders["purchase_month"].value_counts()

purchase_month
August       10843
May          10573
July         10318
March         9893
June          9412
April         9343
February      8508
January       8069
November      7544
December      5674
October       4959
September     4305
Name: count, dtype: int64

In [84]:
orders["purchase_month"].isnull().sum()

0

### Feature 9: purchase_year

Although the dataset only spans a few years, this is still a standard feature in analytics datasets.

**Business Objective**

Identify the year in which the order was placed.

In [85]:
#Python Implementation
orders["purchase_year"] = (
    orders["order_purchase_timestamp"]
    .dt.year
)

In [86]:
orders["purchase_year"].value_counts().sort_index()

purchase_year
2016      329
2017    45101
2018    54011
Name: count, dtype: int64

In [87]:
orders["purchase_year"].isnull().sum()

0

### Feature 10: purchase_quarter
**Business Objective**

Identify the calendar quarter in which the customer placed the order.

In [88]:
#Python Implementation
orders["purchase_quarter"] = (
    "Q" + orders["order_purchase_timestamp"].dt.quarter.astype(str)
)

In [90]:
orders["purchase_quarter"].value_counts().sort_index()

purchase_quarter
Q1    26470
Q2    29328
Q3    25466
Q4    18177
Name: count, dtype: int64

In [91]:
orders["purchase_quarter"].isnull().sum()

0

### Feature 11: purchase_day_of_week
**Business Objective**

Extract the day of the week on which the customer placed the order.

In [92]:
#Python Implementation
orders["purchase_day_of_week"] = (
    orders["order_purchase_timestamp"]
    .dt.day_name()
)

In [93]:
orders["purchase_day_of_week"].value_counts()

purchase_day_of_week
Monday       16196
Tuesday      15963
Wednesday    15552
Thursday     14761
Friday       14122
Sunday       11960
Saturday     10887
Name: count, dtype: int64

In [94]:
orders["purchase_day_of_week"].isnull().sum()

0

### Feature 12: purchase_hour
**Business Objective**

Extract the hour of the day when the customer placed the order.

In [96]:
#Python Implementation
orders["purchase_hour"] = (
    orders["order_purchase_timestamp"]
    .dt.hour
)

In [97]:
orders["purchase_hour"].value_counts().sort_index()

purchase_hour
0     2394
1     1170
2      510
3      272
4      206
5      188
6      502
7     1231
8     2967
9     4785
10    6177
11    6578
12    5995
13    6518
14    6569
15    6454
16    6675
17    6150
18    5769
19    5982
20    6193
21    6217
22    5816
23    4123
Name: count, dtype: int64

In [98]:
orders["purchase_hour"].isnull().sum()

0

### Feature 13: is_weekend_order
**Business Objective**

Identify whether an order was placed on a weekend (Saturday or Sunday).

In [100]:
#Python Implementation
orders["is_weekend_order"] = (
    orders["order_purchase_timestamp"]
    .dt.dayofweek
    .isin([5, 6])
)

In [101]:
orders["is_weekend_order"].value_counts()

is_weekend_order
False    76594
True     22847
Name: count, dtype: int64

In [102]:
orders["is_weekend_order"].value_counts(normalize=True) * 100

is_weekend_order
False    77.024567
True     22.975433
Name: proportion, dtype: float64

In [103]:
orders["is_weekend_order"].isnull().sum()

0

### Feature 14: purchase_year_month
**Business Objective**

Create a Year-Month feature representing the month in which the customer placed the order.

In [104]:
#Python Implementation
orders["purchase_year_month"] = (
    orders["order_purchase_timestamp"]
    .dt.strftime("%Y-%m")
)

In [105]:
orders["purchase_year_month"].value_counts().sort_index()

purchase_year_month
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Name: count, dtype: int64

In [106]:
orders["purchase_year_month"].isnull().sum()

0

### Feature 14: purchase_year_month
**Business Objective**

Create a Year-Month feature representing the month in which the customer placed the order.

In [107]:
#Python Implementation
orders["purchase_year_month"] = (
    orders["order_purchase_timestamp"]
    .dt.strftime("%Y-%m")
)

In [108]:
orders["purchase_year_month"].value_counts().sort_index()

purchase_year_month
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Name: count, dtype: int64

In [109]:
orders["purchase_year_month"].isnull().sum()

0

In [119]:
# Saving feature-engineered orders table

orders.to_csv("../Output/orders_features.csv",index=False)